importar credencailes

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_BUCKET = os.getenv("MINIO_BUCKET_ICEBERG")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")

AZURE_STORAGE_ACCOUNT = os.getenv("AZURE_STORAGE_ACCOUNT")
AZURE_STORAGE_KEY = os.getenv("AZURE_STORAGE_KEY")
AZURE_CONTAINER = os.getenv("AZURE_CONTAINER")
AZURE_FOLDER = os.getenv("AZURE_FOLDER")

In [3]:
import dlt
from dlt.sources.filesystem import readers

pipeline = dlt.pipeline(
    pipeline_name="minio_to_azure",
    destination="filesystem",
    dataset_name="nyc"
)

# leer parquet desde MinIO
parquet_reader = readers(
    bucket_url="s3://iceberg/nyc/",
    file_glob="**/*.parquet"
).read_parquet()

# nombre de tabla
parquet_reader = parquet_reader.with_name("yellow_tripdata")

# ejecutar pipeline
load_info = pipeline.run(
    parquet_reader,
    loader_file_format="parquet",
    write_disposition="replace"
)

print(load_info)

Pipeline minio_to_azure load step completed in 1 minute and 43.68 seconds
1 load package(s) were loaded to destination filesystem and into dataset nyc
The filesystem destination used abfss://clase-4-dlt@fhbd.dfs.core.windows.net/GRUPO_5 location to store data
Load package 1773158635.1636364 is LOADED and contains no failed jobs


In [4]:
import adlfs

fs = adlfs.AzureBlobFileSystem(
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)

print(fs.ls("clase-4-dlt/GRUPO_5/nyc"))

['clase-4-dlt/GRUPO_5/nyc/_dlt_loads', 'clase-4-dlt/GRUPO_5/nyc/_dlt_pipeline_state', 'clase-4-dlt/GRUPO_5/nyc/_dlt_version', 'clase-4-dlt/GRUPO_5/nyc/init', 'clase-4-dlt/GRUPO_5/nyc/yellow_tripdata', 'clase-4-dlt/GRUPO_5/nyc/yellow_tripdata_parquet']


In [ ]:
import adlfs

fs = adlfs.AzureBlobFileSystem(
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)

fs.ls("clase-4-dlt/GRUPO_5/nyc/yellow_tripdata")

['clase-4-dlt/GRUPO_5/nyc/yellow_tripdata/1773158635.1636364.7fa98d8467.parquet']


In [11]:
import pyarrow.parquet as pq
import fsspec

fs = fsspec.filesystem(
    "abfs",
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)

path = "clase-4-dlt/GRUPO_5/nyc/yellow_tripdata/1773158635.1636364.7fa98d8467.parquet"

with fs.open(path, "rb") as f:
    table = pq.read_table(f)

print("Filas:", table.num_rows)

Filas: 6950452
